# 🎙️ Generate a Small Podcast from Notes
**Mini-Project | Podcast Studio | Week 3**

Este notebook transforma suas anotações em um podcast em áudio usando **só a chave da OpenAI**!

---
### Pipeline
```
📄 Suas Notas  →  🤖 GPT-4o (OpenAI)  →  📝 Roteiro  →  🔊 TTS (OpenAI)  →  🎧 Áudio MP3
```

## ⚙️ Step 1: Instalar as bibliotecas
Rode esta célula **uma única vez**. Depois não precisa rodar de novo.

In [ ]:
!pip install openai gradio

## 🔑 Step 2: Configurar a chave da OpenAI
Substitua `sk-...` pela sua chave real (a que sua professora te deu).

In [ ]:
import os
import openai
import gradio as gr
from pathlib import Path

# ✏️ COLOQUE SUA CHAVE AQUI (substitua o sk-... pela chave real)
os.environ["OPENAI_API_KEY"] = "sk-..."

client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("✅ Chave configurada com sucesso!")

## 🤖 Step 3: Transformar notas em roteiro de podcast
Aqui o **GPT-4o** lê suas anotações e reescreve como um roteiro de podcast falado.

In [ ]:
def transform_notes_to_script(notes: str, tone: str = "amigável e educativo") -> str:
    """
    Recebe anotações em texto e retorna um roteiro de podcast pronto.
    """
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": (
                    "Você é um roteirista experiente de podcasts. "
                    "Sua missão é transformar anotações em um roteiro de podcast falado, "
                    "como se um apresentador estivesse falando diretamente com o ouvinte. "
                    "Regras:\n"
                    "- Escreva de forma conversacional, sem bullet points ou markdown.\n"
                    "- Inclua uma introdução calorosa e uma conclusão com resumo.\n"
                    "- Mantenha entre 200 e 400 palavras.\n"
                    "- Use apenas as informações das anotações, não invente fatos."
                )
            },
            {
                "role": "user",
                "content": f"Transforme as notas abaixo em um roteiro de podcast com tom {tone}.\n\nNOTAS:\n{notes}"
            }
        ],
        max_tokens=1024
    )

    script = response.choices[0].message.content
    return script


# --- Teste rápido ---
sample_notes = """
Fotossíntese: plantas usam luz solar, água e CO2 para produzir glicose e oxigênio.
Acontece nos cloroplastos. A clorofila absorve luz vermelha e azul, reflete verde.
Duas etapas: reações de luz (tilacoides) e ciclo de Calvin (estroma).
"""

script = transform_notes_to_script(sample_notes)
print("📝 ROTEIRO GERADO\n" + "="*50)
print(script)

## 🔊 Step 4: Converter roteiro em áudio
Aqui o **OpenAI TTS** lê o roteiro em voz alta e salva como arquivo MP3.

In [ ]:
def generate_audio(script: str, output_path: str = "podcast_output.mp3", voice: str = "nova") -> str:
    """
    Converte o roteiro em um arquivo MP3 usando a OpenAI TTS.
    Vozes disponíveis: alloy, echo, fable, onyx, nova, shimmer
    """
    response = client.audio.speech.create(
        model="tts-1",
        voice=voice,
        input=script
    )

    audio_path = Path(output_path)
    response.stream_to_file(audio_path)

    print(f"✅ Áudio salvo em: {audio_path}")
    return str(audio_path)


# --- Teste rápido ---
audio_file = generate_audio(script, output_path="podcast_output.mp3")
print(f"🎧 Arquivo pronto: {audio_file}")

## 🔗 Step 5: Pipeline completo
Esta função junta tudo: você passa as notas e ela devolve o roteiro + o áudio.

In [ ]:
def generate_podcast_from_notes(notes: str, tone: str = "amigável e educativo", voice: str = "nova"):
    """
    Pipeline completo: notas → roteiro → áudio.
    Retorna: (roteiro_texto, caminho_do_audio)
    """
    if not notes.strip():
        raise ValueError("As notas não podem estar vazias!")

    print("🤖 Passo 1/2 — Gerando roteiro com GPT-4o...")
    script = transform_notes_to_script(notes, tone=tone)

    print("🔊 Passo 2/2 — Gerando áudio com OpenAI TTS...")
    audio_path = generate_audio(script, output_path="podcast_output.mp3", voice=voice)

    print("🎉 Podcast pronto!")
    return script, audio_path


# --- Teste do pipeline completo ---
script, audio = generate_podcast_from_notes(sample_notes)
print("\n📝 Prévia do roteiro:")
print(script[:200] + "...")

## 🖥️ Step 6: Interface Gradio
Rode esta célula e uma tela bonita vai abrir no seu browser para usar o podcast gerador!

In [ ]:
def gradio_podcast_generator(notes: str, tone: str, voice: str):
    try:
        script, audio_path = generate_podcast_from_notes(notes, tone=tone, voice=voice)
        return script, audio_path
    except Exception as e:
        return f"❌ Erro: {str(e)}", None


with gr.Blocks(title="🎙️ Podcast das Notas", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🎙️ Podcast das Notas\nCole suas anotações abaixo e gere um podcast em áudio!")

    with gr.Row():
        with gr.Column():
            notes_input = gr.Textbox(
                label="📄 Suas Anotações",
                placeholder="Cole aqui suas notas de aula, artigo, texto...",
                lines=10
            )
            with gr.Row():
                tone_input = gr.Dropdown(
                    label="🎭 Tom do Podcast",
                    choices=[
                        "amigável e educativo",
                        "profissional e direto",
                        "casual e divertido",
                        "narrativo e envolvente"
                    ],
                    value="amigável e educativo"
                )
                voice_input = gr.Dropdown(
                    label="🗣️ Voz",
                    choices=["nova", "alloy", "echo", "fable", "onyx", "shimmer"],
                    value="nova"
                )
            generate_btn = gr.Button("🎙️ Gerar Podcast", variant="primary")

        with gr.Column():
            script_output = gr.Textbox(
                label="📝 Roteiro Gerado",
                lines=12,
                interactive=False
            )
            audio_output = gr.Audio(
                label="🎧 Ouça seu Podcast",
                type="filepath"
            )

    generate_btn.click(
        fn=gradio_podcast_generator,
        inputs=[notes_input, tone_input, voice_input],
        outputs=[script_output, audio_output]
    )

    gr.Examples(
        examples=[
            [sample_notes, "amigável e educativo", "nova"]
        ],
        inputs=[notes_input, tone_input, voice_input]
    )

demo.launch(share=False)

---
## ✅ Resumo do que cada célula faz

| Célula | O que faz |
|--------|----------|
| Step 1 | Instala as bibliotecas necessárias |
| Step 2 | Configura sua chave da OpenAI |
| Step 3 | GPT-4o transforma suas notas em roteiro |
| Step 4 | TTS transforma o roteiro em áudio MP3 |
| Step 5 | Pipeline completo (tudo junto) |
| Step 6 | Abre a tela Gradio no browser |